# Rule-Based Counterattack Extraction

This notebook applies the final rule-based definition used to identify counterattacks from synchronized event and tracking data.

The pipeline extracts:
1. `counterattack attempt` after a possession gain
2. `counterattack success` outcomes such as shots, box entries and final-third entries

The extracted attempts are saved in `Data/derived/` and are used later as rule-based labels for the XGBoost models.


## Imports
Load the packages and tracking-direction helper used in the extraction pipeline.


In [1]:
from __future__ import annotations
import csv
import math
from bisect import bisect_left, bisect_right
from collections import defaultdict
from pathlib import Path
import pandas as pd
from tracking_direction import build_tracking_direction_lookup, lookup_attacking_left


## Input Paths and Definition Parameters
Define the event input file, on-ball event types, pitch references and the final rule thresholds.


In [2]:
DATA_PATH = Path('Data/events.csv')
assert DATA_PATH.exists(), f'Missing file: {DATA_PATH}'
ON_BALL_TYPES = {
    'pass', 'touch', 'duel', 'interception', 'shot', 'clearance',
    'free_kick', 'throw_in', 'corner', 'goal_kick', 'goalkeeper_exit',
    'penalty', 'postmatch_penalty', 'postmatch_penalty_faced', 'own_goal'
}

# Event space goal reference (x in [0,100], y in [0,100]).
GOAL_X_EVENT = 100.0
GOAL_Y_EVENT = 50.0
# Approx. max distance to goal center in meters from far corner (0,0).
MAX_GOAL_DISTANCE_M = math.hypot(GOAL_X_EVENT * 1.05, GOAL_Y_EVENT * 0.68)
# Distance from center line center to goal center in event space (50,50 -> 100,50).
MIDFIELD_GOAL_DISTANCE_M = (GOAL_X_EVENT - 50.0) * 1.05

# The definition is kept in one dictionary so the main assumptions are easy to inspect.
# 17s is the reference maximum duration when the regain starts around midfield.
# The allowed time/pass limits are then scaled by how far the ball starts from goal.
# min_duration_s is a lower bound on the scaled maximum duration, not a hard upper limit.
# min_end_progress_m ensures that the attack reaches at least the opponent half.
V2_BALANCED_BASE = {
    'reference_goal_distance_m': MIDFIELD_GOAL_DISTANCE_M,
    'reference_max_duration_s': 17.0,
    'reference_max_passes': 5,
    'min_duration_s': 2.5,
    'min_in_play_s': 2.5,
    'speed_margin': 1.0,
    'min_required_speed_kmh': 12.5,
    'opponent_byline_required_speed_kmh': 9.0,
    'required_speed_scale_start_m': 52.5,
    'required_speed_scale_end_m': 105.0,
    'min_end_progress_m': 52.5,
    'min_duration_progression_scale': {
        'own_line_s': 4.0,
        'midfield_s': 3.25,
        'opponent_line_s': 2.5,
    },
}

STOP_RESTART_TYPES = {'throw_in', 'goal_kick', 'corner'}




## Utility Functions
Helper functions for parsing event fields, detecting possession gains, reading tracking context and converting events into pitch-based metrics.


In [3]:
def to_float(value):
    s = (value or '').strip()
    if not s or s == 'NULL':
        return None
    s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return None

def to_int(value):
    x = to_float(value)
    return int(x) if x is not None else None

def ts_to_seconds(ts):
    s = (ts or '').strip()
    if not s:
        return None
    h, m, sec = s.split(':')
    return int(h) * 3600 + int(m) * 60 + float(sec)

def start_zone(x):
    if x is None:
        return 'unknown'
    if x < 33:
        return 'defensive_third'
    if x < 66:
        return 'middle_third'
    return 'attacking_third'

def distance_to_goal_m(x, y, goal_x):
    if x is None or y is None or goal_x is None:
        return None
    dx = (goal_x - x) * 1.05
    dy = (50.0 - y) * 0.68
    return math.hypot(dx, dy)

def duel_won(row):
    return (
        (row.get('aerialWon') or '').strip() == '1'
        or (row.get('groundDuelRecoveredPossession') or '').strip() == '1'
        or (row.get('groundDuelKeptPossession') or '').strip() == '1'
    )

def is_regain_event(row):
    t = (row.get('typePrimary') or '').strip()
    if t == 'interception':
        return True
    if t == 'duel' and duel_won(row):
        return True
    return False


# The possession should only end when the opponent actually controls an on-ball action.
# This avoids ending a sequence just because the opponent is involved in a duel they do not win.
def opponent_on_ball_ends_sequence(row, possessing_team_id):
    t = (row.get('typePrimary') or '').strip()
    team = (row.get('teamId') or '').strip()
    if not team or team == possessing_team_id or t not in ON_BALL_TYPES:
        return False
    # Opponent duels should only end the sequence if the opponent actually wins/keeps possession.
    if t == 'duel':
        return duel_won(row)
    return True


def normalize_label_no_score(label):
    s = (label or '').strip()
    if ',' in s:
        s = s.split(',', 1)[0].strip()
    return s


def build_tracking_label_index(root=Path('Data')):
    label_to_folder = {}
    for comp in ['H_EURO2024', 'Q_EURO2025', 'U21_EURO2025']:
        comp_path = root / comp
        if not comp_path.exists():
            continue
        for d in sorted(comp_path.iterdir()):
            if not d.is_dir():
                continue
            home = d / 'home.csv'
            if not home.exists():
                continue
            with home.open(newline='', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                first = next(reader, None)
                if not first:
                    continue
                lbl = normalize_label_no_score(first.get('label'))
                if lbl:
                    label_to_folder[lbl] = d
    return label_to_folder


def build_ball_in_play_intervals(by_match, label_to_folder, pitch_x_max=52.5, pitch_y_max=34.0):
    out = {}
    for match_id, rows in by_match.items():
        if not rows:
            continue
        lbl = normalize_label_no_score(rows[0].get('label'))
        folder = label_to_folder.get(lbl)
        if folder is None:
            continue
        home_csv = folder / 'home.csv'
        if not home_csv.exists():
            continue

        intervals = []
        state = None
        start_t = None
        end_t = None
        with home_csv.open(newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                t = to_float(row.get('total_time_passed'))
                if t is None:
                    continue
                bx = to_float(row.get('ball_x'))
                by = to_float(row.get('ball_y'))
                in_play = (
                    bx is not None and by is not None
                    and abs(bx) <= pitch_x_max
                    and abs(by) <= pitch_y_max
                )
                if state is None:
                    state = in_play
                    start_t = t
                    end_t = t
                elif in_play == state:
                    end_t = t
                else:
                    intervals.append((start_t, end_t, state))
                    state = in_play
                    start_t = t
                    end_t = t
        if state is not None:
            intervals.append((start_t, end_t, state))

        if intervals:
            out[match_id] = {
                'intervals': intervals,
                'starts': [it[0] for it in intervals],
            }
    return out


def compute_in_play_duration_s(ball_state_by_match, match_id, start_ts, end_ts):
    if start_ts is None or end_ts is None or end_ts <= start_ts:
        return 0.0
    if ball_state_by_match is None:
        return max(0.0, end_ts - start_ts)

    data = ball_state_by_match.get(match_id)
    if not data:
        return max(0.0, end_ts - start_ts)

    total = 0.0
    for s, e, state in data['intervals']:
        if e <= start_ts:
            continue
        if s >= end_ts:
            break
        if not state:
            continue
        overlap = max(0.0, min(e, end_ts) - max(s, start_ts))
        total += overlap
    return total


def build_ball_tracking_lookup(by_match, label_to_folder):
    out = {}
    for match_id, rows in by_match.items():
        if not rows:
            continue
        lbl = normalize_label_no_score(rows[0].get('label'))
        folder = label_to_folder.get(lbl)
        if folder is None:
            continue
        home_csv = folder / 'home.csv'
        if not home_csv.exists():
            continue

        times = []
        xs = []
        ys = []
        with home_csv.open(newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                t = to_float(row.get('total_time_passed'))
                bx = to_float(row.get('ball_x'))
                by = to_float(row.get('ball_y'))
                if t is None or bx is None or by is None:
                    continue
                times.append(t)
                xs.append(bx)
                ys.append(by)
        if times:
            out[match_id] = {
                'times': times,
                'xs': xs,
                'ys': ys,
            }
    return out


TRACKING_GOAL_LEFT = (-52.5, 0.0)
TRACKING_GOAL_RIGHT = (52.5, 0.0)
TRACKING_FINAL_THIRD_PROGRESS_M = 70.0
TRACKING_BOX_X_LIMIT = 52.5 - 16.5
TRACKING_BOX_HALF_WIDTH = 20.16


def goal_distance_tracking(x, y, goal_xy):
    return math.hypot(x - goal_xy[0], y - goal_xy[1])


def forward_progress_m(x, attacking_left):
    return (52.5 - x) if attacking_left else (x + 52.5)


def tracking_start_zone(progress_m):
    if progress_m is None:
        return 'unknown'
    if progress_m < 35.0:
        return 'defensive_third'
    if progress_m < 70.0:
        return 'middle_third'
    return 'attacking_third'


def tracking_box_entry(x, y, attacking_left):
    if x is None or y is None:
        return False
    if attacking_left:
        return x <= -TRACKING_BOX_X_LIMIT and abs(y) <= TRACKING_BOX_HALF_WIDTH
    return x >= TRACKING_BOX_X_LIMIT and abs(y) <= TRACKING_BOX_HALF_WIDTH


def nearest_tracking_ball_position(ball_tracking_by_match, match_id, ts, max_dt_s=0.25):
    if ball_tracking_by_match is None or ts is None:
        return None
    data = ball_tracking_by_match.get(match_id)
    if not data:
        return None

    times = data['times']
    xs = data['xs']
    ys = data['ys']
    idx = bisect_right(times, ts)
    candidates = []
    if idx > 0:
        candidates.append(idx - 1)
    if idx < len(times):
        candidates.append(idx)
    if not candidates:
        return None

    best_idx = min(candidates, key=lambda i: abs(times[i] - ts))
    dt = abs(times[best_idx] - ts)
    if dt > max_dt_s:
        return None
    return {
        't': times[best_idx],
        'x': xs[best_idx],
        'y': ys[best_idx],
        'dt': dt,
    }


def get_tracking_ball_window(ball_tracking_by_match, match_id, start_ts, end_ts):
    if ball_tracking_by_match is None:
        return []
    data = ball_tracking_by_match.get(match_id)
    if not data:
        return []
    times = data['times']
    xs = data['xs']
    ys = data['ys']
    lo = bisect_left(times, start_ts)
    hi = bisect_right(times, end_ts)
    return list(zip(times[lo:hi], xs[lo:hi], ys[lo:hi]))


def infer_attacking_left(track_points):
    if len(track_points) < 2:
        return None
    left_d = [goal_distance_tracking(x, y, TRACKING_GOAL_LEFT) for _, x, y in track_points]
    right_d = [goal_distance_tracking(x, y, TRACKING_GOAL_RIGHT) for _, x, y in track_points]
    left_prog = left_d[0] - min(left_d)
    right_prog = right_d[0] - min(right_d)
    if abs(left_prog - right_prog) < 1e-6:
        dx = track_points[-1][1] - track_points[0][1]
        return dx < 0
    return left_prog > right_prog


def infer_attacking_left_from_start(ball_tracking_by_match, match_id, ts, lookahead_s=2.0):
    pts = get_tracking_ball_window(ball_tracking_by_match, match_id, ts, ts + lookahead_s)
    if len(pts) < 2:
        return None
    return infer_attacking_left(pts)


def compute_tracking_sequence_features(track_points, attacking_left):
    if len(track_points) < 2 or attacking_left is None:
        return None

    goal_xy = TRACKING_GOAL_LEFT if attacking_left else TRACKING_GOAL_RIGHT
    start_x_raw = track_points[0][1]
    start_y_raw = track_points[0][2]
    start_progress_m = forward_progress_m(start_x_raw, attacking_left)

    progress_vals = [forward_progress_m(x, attacking_left) for _, x, _ in track_points]
    max_progress_m = max(progress_vals)

    goal_dists = [goal_distance_tracking(x, y, goal_xy) for _, x, y in track_points]
    start_goal_distance_m = goal_dists[0]
    min_goal_distance_m = min(goal_dists)
    progression_goal_m = max(0.0, start_goal_distance_m - min_goal_distance_m)

    sum_abs_goal_delta = 0.0
    for prev_d, next_d in zip(goal_dists[:-1], goal_dists[1:]):
        sum_abs_goal_delta += abs(next_d - prev_d)
    directness_goal = None
    if sum_abs_goal_delta > 0:
        directness_goal = progression_goal_m / sum_abs_goal_delta

    progression_x = max(0.0, max_progress_m - start_progress_m)
    end_progress_m = progress_vals[-1]
    final_third_entry = any(p >= TRACKING_FINAL_THIRD_PROGRESS_M for p in progress_vals)
    box_entry = any(tracking_box_entry(x, y, attacking_left) for _, x, y in track_points)

    return {
        'tracking_start_x_m_raw': start_x_raw,
        'tracking_start_y_m_raw': start_y_raw,
        'start_x': start_progress_m,
        'max_x': max_progress_m,
        'end_x': end_progress_m,
        'progression_x': progression_x,
        'start_zone': tracking_start_zone(start_progress_m),
        'start_goal_distance_m': start_goal_distance_m,
        'min_goal_distance_m': min_goal_distance_m,
        'progression_goal_m': progression_goal_m,
        'directness_goal': directness_goal,
        'success_secondary_box_entry': int(box_entry),
        'success_secondary_final_third_entry': int(final_third_entry),
    }



## Load Event and Tracking Context
Load the event data by match and prepare the tracking lookups needed for ball-in-play intervals, ball positions and fixed attacking directions.


In [4]:
def load_events(path=DATA_PATH):
    by_match = defaultdict(list)
    with path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            match_id = (row.get('matchId') or '').strip()
            if not match_id:
                continue
            row['_ts'] = ts_to_seconds(row.get('matchTimestamp'))
            row['_event_id'] = to_int(row.get('eventId')) or 0
            row['_x'] = to_float(row.get('x'))
            row['_y'] = to_float(row.get('y'))
            by_match[match_id].append(row)

    for match_id in by_match:
        by_match[match_id].sort(key=lambda r: ((r['_ts'] if r['_ts'] is not None else 10**18), r['_event_id']))
    return by_match


by_match_events = load_events()
label_to_tracking_folder = build_tracking_label_index()
ball_state_by_match = build_ball_in_play_intervals(by_match_events, label_to_tracking_folder)
ball_tracking_by_match = build_ball_tracking_lookup(by_match_events, label_to_tracking_folder)
tracking_direction_by_match = build_tracking_direction_lookup(by_match_events, label_to_tracking_folder)
print(f'Matches loaded: {len(by_match_events)}')
print(f'Events loaded: {sum(len(v) for v in by_match_events.values())}')
print(f'Matches with tracking in-play intervals: {len(ball_state_by_match)}')
print(f'Matches with tracking ball lookup: {len(ball_tracking_by_match)}')
print(f'Matches with fixed tracking directions: {len(tracking_direction_by_match)}')



Matches loaded: 113
Events loaded: 191001
Matches with tracking in-play intervals: 106
Matches with tracking ball lookup: 106
Matches with fixed tracking directions: 103


## Counterattack Extraction Logic
Core extraction logic. Starting from possession regains, the pipeline checks duration, in-play time, pass count, progression, speed, end position and outcome conditions.


In [ ]:
def min_duration_for_zone(base_cfg, zone=None):
    zone_cfg = base_cfg.get('zone_min_duration_s') or {}
    if zone is not None and zone in zone_cfg:
        return zone_cfg[zone]
    return base_cfg['min_duration_s']


def min_duration_for_progress(base_cfg, start_progress_m=None, zone=None):
    scale_cfg = base_cfg.get('min_duration_progression_scale') or {}
    own_line_s = scale_cfg.get('own_line_s')
    midfield_s = scale_cfg.get('midfield_s')
    opponent_line_s = scale_cfg.get('opponent_line_s')

    if own_line_s is None or midfield_s is None or opponent_line_s is None or start_progress_m is None:
        return min_duration_for_zone(base_cfg, zone)

    # Tracking progress runs from 0m at own byline to 105m at opponent byline.
    # Attacks that start close to the opponent goal are allowed to be shorter,
    # while attacks starting deep in own half need slightly more time to develop.
    p = max(0.0, min(105.0, start_progress_m))
    if p <= 52.5:
        frac = p / 52.5
        return own_line_s + frac * (midfield_s - own_line_s)

    frac = (p - 52.5) / 52.5
    return midfield_s + frac * (opponent_line_s - midfield_s)


def min_required_speed_floor_mps(base_cfg, start_progress_m=None):
    # Close to the opponent byline, the required goal-directed speed is relaxed.
    # Otherwise short high-field regains would be rejected for not covering enough distance quickly.
    base_mps = (base_cfg.get('min_required_speed_kmh', 0.0) or 0.0) / 3.6
    opponent_byline_kmh = base_cfg.get('opponent_byline_required_speed_kmh')
    if opponent_byline_kmh is None or start_progress_m is None:
        return base_mps

    low_mps = opponent_byline_kmh / 3.6
    scale_start_m = base_cfg.get('required_speed_scale_start_m', 52.5)
    scale_end_m = base_cfg.get('required_speed_scale_end_m', 105.0)
    if scale_end_m <= scale_start_m:
        return base_mps

    p = max(0.0, min(scale_end_m, start_progress_m))
    if p <= scale_start_m:
        return base_mps
    if p >= scale_end_m:
        return low_mps

    frac = (p - scale_start_m) / (scale_end_m - scale_start_m)
    return base_mps + frac * (low_mps - base_mps)


def scaled_requirements_v2(start_goal_distance_m, base_cfg, start_progress_m=None):
    ref_d = base_cfg['reference_goal_distance_m']
    if start_goal_distance_m is None or ref_d <= 0:
        return None

    # Scale the reference maximum duration and pass count by distance to goal.
    # A regain far from goal is allowed more time than a regain already high up the pitch.
    distance_factor = start_goal_distance_m / ref_d
    scaled_max_duration_s = base_cfg['reference_max_duration_s'] * distance_factor
    scaled_max_duration_s = max(base_cfg['min_duration_s'], scaled_max_duration_s)
    scaled_max_duration_s = min(base_cfg.get('max_duration_s_cap', scaled_max_duration_s), scaled_max_duration_s)
    scaled_max_passes = max(1, int(math.ceil(base_cfg['reference_max_passes'] * distance_factor)))

    required_speed_mps = start_goal_distance_m / scaled_max_duration_s
    min_required_speed_mps = min_required_speed_floor_mps(base_cfg, start_progress_m)
    required_speed_mps = max(required_speed_mps, min_required_speed_mps)

    return {
        'ratio_to_max': max(0.0, min(1.0, start_goal_distance_m / MAX_GOAL_DISTANCE_M)),
        'distance_factor': distance_factor,
        'max_duration_s': scaled_max_duration_s,
        'max_passes': scaled_max_passes,
        'required_speed_mps': required_speed_mps,
    }


def extract_attempts_v2_scaled(profile_name, base_cfg, by_match, ball_state_by_match=None):
    out = []
    for match_id, rows in by_match.items():
        prev_control_team = None
        for i, ev in enumerate(rows):
            t = (ev.get('typePrimary') or '').strip()
            team = (ev.get('teamId') or '').strip()
            ts = ev['_ts']
            if t in ON_BALL_TYPES and team:
                candidate = (
                    is_regain_event(ev)
                    and prev_control_team is not None
                    and team != prev_control_team
                    and ts is not None
                )
                if candidate:
                    start_x = ev['_x']
                    start_y = ev['_y']
                    start_goal_distance_m = distance_to_goal_m(start_x, start_y, GOAL_X_EVENT)
                    scaled = scaled_requirements_v2(start_goal_distance_m, base_cfg)
                    if scaled is None:
                        if t != 'duel' or duel_won(ev):
                            prev_control_team = team
                        continue

                    max_x = start_x
                    pass_count = 0
                    action_count = 0
                    sum_abs_goal_delta = 0.0
                    prev_goal_distance_m = start_goal_distance_m
                    min_goal_distance_m = start_goal_distance_m

                    success_primary_shot = 0
                    success_primary_shot_box = 0
                    goal_count = 0
                    success_secondary_box_entry = 0
                    success_secondary_final_third_entry = 0
                    end_ts = ts
                    for j in range(i + 1, len(rows)):
                        ev2 = rows[j]
                        t2 = (ev2.get('typePrimary') or '').strip()
                        team2 = (ev2.get('teamId') or '').strip()
                        ts2 = ev2['_ts']
                        if ts2 is None:
                            continue
                        if ts2 - ts > scaled['max_duration_s']:
                            break
                        if t2 == 'game_interruption':
                            break
                        if t2 in STOP_RESTART_TYPES:
                            break
                        if opponent_on_ball_ends_sequence(ev2, team):
                            break

                        if team2 == team:
                            end_ts = ts2
                            action_count += 1
                            if t2 == 'pass':
                                pass_count += 1

                            x2 = ev2['_x']
                            y2 = ev2['_y']
                            if x2 is not None:
                                if max_x is None or x2 > max_x:
                                    max_x = x2
                                if x2 >= 83:
                                    success_secondary_box_entry = 1
                                if x2 >= 66:
                                    success_secondary_final_third_entry = 1

                            d2 = distance_to_goal_m(x2, y2, GOAL_X_EVENT)
                            if d2 is not None:
                                if min_goal_distance_m is None or d2 < min_goal_distance_m:
                                    min_goal_distance_m = d2
                                if prev_goal_distance_m is not None:
                                    sum_abs_goal_delta += abs(d2 - prev_goal_distance_m)
                                prev_goal_distance_m = d2

                            if t2 == 'shot':
                                success_primary_shot = 1
                                if (ev2.get('isGoal') or '').strip() == '1':
                                    goal_count += 1
                                if x2 is not None and x2 >= 83:
                                    success_primary_shot_box = 1

                    duration_s = max(0.0, end_ts - ts)
                    progression_x = None
                    if start_x is not None and max_x is not None:
                        progression_x = max_x - start_x

                    progression_goal_m = None
                    if start_goal_distance_m is not None and min_goal_distance_m is not None:
                        progression_goal_m = max(0.0, start_goal_distance_m - min_goal_distance_m)

                    directness_goal = None
                    if progression_goal_m is not None and sum_abs_goal_delta > 0:
                        directness_goal = progression_goal_m / sum_abs_goal_delta

                    avg_progress_speed_mps = None
                    if progression_goal_m is not None and duration_s > 0:
                        avg_progress_speed_mps = progression_goal_m / duration_s

                    progress_fraction = None
                    if progression_goal_m is not None and start_goal_distance_m and start_goal_distance_m > 0:
                        progress_fraction = progression_goal_m / start_goal_distance_m

                    time_fraction = None
                    if scaled['max_duration_s'] > 0:
                        time_fraction = duration_s / scaled['max_duration_s']

                    speed_ratio = None
                    if avg_progress_speed_mps is not None and scaled['required_speed_mps'] > 0:
                        speed_ratio = avg_progress_speed_mps / scaled['required_speed_mps']

                    in_play_duration_s = compute_in_play_duration_s(ball_state_by_match, match_id, ts, end_ts)

                    required_min_duration_s = min_duration_for_progress(base_cfg, track_features['start_x'], track_features['start_zone'])

                    keep = True
                    if duration_s < required_min_duration_s:
                        keep = False
                    if in_play_duration_s < base_cfg.get('min_in_play_s', 0.0):
                        keep = False
                    if pass_count > scaled['max_passes']:
                        keep = False
                    if avg_progress_speed_mps is None or avg_progress_speed_mps < base_cfg['speed_margin'] * scaled['required_speed_mps']:
                        keep = False
                    if base_cfg.get('min_directness', 0.0) > 0 and (directness_goal is None or directness_goal < base_cfg['min_directness']):
                        keep = False

                    if keep:
                        out.append({
                            'profile': profile_name,
                            'coordinate_source': 'event',
                            'match_id': match_id,
                            'team_id': team,
                            'match_period': (ev.get('matchPeriod') or '').strip(),
                            'start_event_id': ev['_event_id'],
                            'start_type': (ev.get('typePrimary') or '').strip(),
                            'start_ts': ts,
                            'end_ts': end_ts,
                            'duration_s': duration_s,
                            'in_play_duration_s': in_play_duration_s,
                            'start_x': start_x,
                            'max_x': max_x,
                            'progression_x': progression_x,
                            'directness': directness_goal,
                            'pass_count': pass_count,
                            'action_count': action_count,
                            'start_zone': start_zone(start_x),
                            'success_primary_shot': success_primary_shot,
                            'success_primary_shot_box': success_primary_shot_box,
                            'goal_count': goal_count,
                            'success_secondary_box_entry': success_secondary_box_entry,
                            'success_secondary_final_third_entry': success_secondary_final_third_entry,
                            'start_goal_distance_m': start_goal_distance_m,
                            'min_goal_distance_m': min_goal_distance_m,
                            'progression_goal_m': progression_goal_m,
                            'avg_progress_speed_mps': avg_progress_speed_mps,
                            'required_speed_mps': scaled['required_speed_mps'],
                            'speed_ratio': speed_ratio,
                            'progress_fraction': progress_fraction,
                            'time_fraction': time_fraction,
                            'scaled_ratio_to_max': scaled['ratio_to_max'],
                            'scaled_distance_factor': scaled['distance_factor'],
                            'scaled_max_duration_s': scaled['max_duration_s'],
                            'scaled_max_passes': scaled['max_passes'],
                        })

                if t != 'duel' or duel_won(ev):
                    prev_control_team = team
    return out


def extract_attempts_v2_scaled_tracking_coords(profile_name, base_cfg, by_match, ball_state_by_match=None, ball_tracking_by_match=None, tracking_direction_by_match=None):
    out = []
    for match_id, rows in by_match.items():
        prev_control_team = None
        for i, ev in enumerate(rows):
            t = (ev.get('typePrimary') or '').strip()
            team = (ev.get('teamId') or '').strip()
            ts = ev['_ts']
            if t in ON_BALL_TYPES and team:
                candidate = (
                    is_regain_event(ev)
                    and prev_control_team is not None
                    and team != prev_control_team
                    and ts is not None
                )
                if candidate:
                    start_ball = nearest_tracking_ball_position(ball_tracking_by_match, match_id, ts)
                    # Prefer the fixed half-level direction inferred from the players' starting positions.
                    # The fallback is only used if that direction lookup is unavailable for the match.
                    attacking_left = lookup_attacking_left(tracking_direction_by_match, match_id, team, (ev.get('matchPeriod') or '').strip())
                    if attacking_left is None:
                        attacking_left = infer_attacking_left_from_start(ball_tracking_by_match, match_id, ts)
                    if start_ball is None or attacking_left is None:
                        if t != 'duel' or duel_won(ev):
                            prev_control_team = team
                        continue

                    goal_xy = TRACKING_GOAL_LEFT if attacking_left else TRACKING_GOAL_RIGHT
                    start_goal_distance_m = goal_distance_tracking(start_ball['x'], start_ball['y'], goal_xy)
                    start_progress_m = forward_progress_m(start_ball['x'], attacking_left)
                    scaled = scaled_requirements_v2(start_goal_distance_m, base_cfg, start_progress_m)
                    if scaled is None:
                        if t != 'duel' or duel_won(ev):
                            prev_control_team = team
                        continue

                    pass_count = 0
                    action_count = 0
                    success_primary_shot = 0
                    success_primary_shot_box = 0
                    goal_count = 0
                    end_ts = ts

                    # Scan forward until possession ends, play stops, or the scaled max duration is reached.
                    for j in range(i + 1, len(rows)):
                        ev2 = rows[j]
                        t2 = (ev2.get('typePrimary') or '').strip()
                        team2 = (ev2.get('teamId') or '').strip()
                        ts2 = ev2['_ts']
                        if ts2 is None:
                            continue
                        if ts2 - ts > scaled['max_duration_s']:
                            break
                        if t2 == 'game_interruption':
                            break
                        if t2 in STOP_RESTART_TYPES:
                            break
                        if opponent_on_ball_ends_sequence(ev2, team):
                            break

                        if team2 == team:
                            end_ts = ts2
                            action_count += 1
                            if t2 == 'pass':
                                pass_count += 1

                            if t2 == 'shot':
                                success_primary_shot = 1
                                if (ev2.get('isGoal') or '').strip() == '1':
                                    goal_count += 1
                                shot_ball = nearest_tracking_ball_position(ball_tracking_by_match, match_id, ts2)
                                if shot_ball is not None and tracking_box_entry(shot_ball['x'], shot_ball['y'], attacking_left):
                                    success_primary_shot_box = 1

                    duration_s = max(0.0, end_ts - ts)
                    in_play_duration_s = compute_in_play_duration_s(ball_state_by_match, match_id, ts, end_ts)
                    track_points = get_tracking_ball_window(ball_tracking_by_match, match_id, ts, end_ts)
                    track_features = compute_tracking_sequence_features(track_points, attacking_left)
                    if track_features is None:
                        if t != 'duel' or duel_won(ev):
                            prev_control_team = team
                        continue

                    avg_progress_speed_mps = None
                    if track_features['progression_goal_m'] is not None and duration_s > 0:
                        avg_progress_speed_mps = track_features['progression_goal_m'] / duration_s

                    progress_fraction = None
                    if track_features['progression_goal_m'] is not None and track_features['start_goal_distance_m']:
                        if track_features['start_goal_distance_m'] > 0:
                            progress_fraction = track_features['progression_goal_m'] / track_features['start_goal_distance_m']

                    time_fraction = None
                    if scaled['max_duration_s'] > 0:
                        time_fraction = duration_s / scaled['max_duration_s']

                    speed_ratio = None
                    if avg_progress_speed_mps is not None and scaled['required_speed_mps'] > 0:
                        speed_ratio = avg_progress_speed_mps / scaled['required_speed_mps']

                    required_min_duration_s = min_duration_for_progress(base_cfg, track_features['start_x'], track_features['start_zone'])

                    # Final acceptance gates, all of these conditions must hold for the regain
                    # to be counted as a counterattack by the rule-based definition
                    keep = True
                    if duration_s < required_min_duration_s:
                        keep = False
                    if in_play_duration_s < base_cfg.get('min_in_play_s', 0.0):
                        keep = False
                    if pass_count > scaled['max_passes']:
                        keep = False
                    # Speed is measured as goal-directed progression per second, not raw ball speed.
                    if avg_progress_speed_mps is None or avg_progress_speed_mps < base_cfg['speed_margin'] * scaled['required_speed_mps']:
                        keep = False
                    if base_cfg.get('min_directness', 0.0) > 0 and (track_features['directness_goal'] is None or track_features['directness_goal'] < base_cfg['min_directness']):
                        keep = False
                    # The ball must finish in the opponent half, otherwise the sequence is not treated as a counterattack.
                    if track_features['end_x'] < base_cfg.get('min_end_progress_m', 0.0):
                        keep = False

                    if keep:
                        out.append({
                            'profile': profile_name,
                            'coordinate_source': 'tracking',
                            'match_id': match_id,
                            'team_id': team,
                            'match_period': (ev.get('matchPeriod') or '').strip(),
                            'start_event_id': ev['_event_id'],
                            'start_type': (ev.get('typePrimary') or '').strip(),
                            'start_ts': ts,
                            'end_ts': end_ts,
                            'duration_s': duration_s,
                            'in_play_duration_s': in_play_duration_s,
                            'start_x': track_features['start_x'],
                            'max_x': track_features['max_x'],
                            'end_x': track_features['end_x'],
                            'progression_x': track_features['progression_x'],
                            'directness': track_features['directness_goal'],
                            'pass_count': pass_count,
                            'action_count': action_count,
                            'start_zone': track_features['start_zone'],
                            'success_primary_shot': success_primary_shot,
                            'success_primary_shot_box': success_primary_shot_box,
                            'goal_count': goal_count,
                            'success_secondary_box_entry': track_features['success_secondary_box_entry'],
                            'success_secondary_final_third_entry': track_features['success_secondary_final_third_entry'],
                            'start_goal_distance_m': track_features['start_goal_distance_m'],
                            'min_goal_distance_m': track_features['min_goal_distance_m'],
                            'progression_goal_m': track_features['progression_goal_m'],
                            'avg_progress_speed_mps': avg_progress_speed_mps,
                            'required_speed_mps': scaled['required_speed_mps'],
                            'speed_ratio': speed_ratio,
                            'progress_fraction': progress_fraction,
                            'time_fraction': time_fraction,
                            'scaled_ratio_to_max': scaled['ratio_to_max'],
                            'scaled_distance_factor': scaled['distance_factor'],
                            'scaled_max_duration_s': scaled['max_duration_s'],
                            'scaled_max_passes': scaled['max_passes'],
                            'required_min_duration_s': required_min_duration_s,
                            'tracking_start_x_m_raw': track_features['tracking_start_x_m_raw'],
                            'tracking_start_y_m_raw': track_features['tracking_start_y_m_raw'],
                            'tracking_attacking_left': int(attacking_left),
                            'tracking_start_dt_s': start_ball['dt'],
                        })

                if t != 'duel' or duel_won(ev):
                    prev_control_team = team
    return out



## Run Final Definition
Apply the final rule configuration to all matches and collect the extracted counterattack attempts.


In [6]:
attempt_rows = []

tracking_attempts = extract_attempts_v2_scaled_tracking_coords(
    'balanced_v2_scaled_tracking_coords',
    V2_BALANCED_BASE,
    by_match_events,
    ball_state_by_match,
    ball_tracking_by_match,
    tracking_direction_by_match,
)
attempt_rows.extend(tracking_attempts)

attempts_df = pd.DataFrame(attempt_rows)
attempts_df.shape


(746, 41)

## Definition Summary
Summarise the number of extracted attempts and the main success outcomes.


In [7]:
summary_df = pd.DataFrame([{
    'profile': 'balanced_v2_scaled_tracking_coords',
    'attempts': len(attempts_df),
    'shot_rate_%': round(100 * attempts_df['success_primary_shot'].mean(), 2),
    'shot_box_rate_%': round(100 * attempts_df['success_primary_shot_box'].mean(), 2),
    'goals_total': int(attempts_df['goal_count'].sum()),
    'goal_rate_%': round(100 * (attempts_df['goal_count'].sum() / len(attempts_df)), 2),
    'box_entry_rate_%': round(100 * attempts_df['success_secondary_box_entry'].mean(), 2),
    'final_third_entry_rate_%': round(100 * attempts_df['success_secondary_final_third_entry'].mean(), 2),
    'median_duration_s': round(attempts_df['duration_s'].median(), 2),
    'median_in_play_duration_s': round(attempts_df['in_play_duration_s'].median(), 2),
    'median_progression_x': round(attempts_df['progression_x'].median(), 2),
    'median_passes': round(attempts_df['pass_count'].median(), 2),
    'median_progression_goal_m': round(attempts_df['progression_goal_m'].median(), 2),
    'median_start_goal_distance_m': round(attempts_df['start_goal_distance_m'].median(), 2),
    'median_required_speed_mps': round(attempts_df['required_speed_mps'].median(), 2),
    'median_avg_progress_speed_mps': round(attempts_df['avg_progress_speed_mps'].median(), 2),
    'median_speed_ratio': round(attempts_df['speed_ratio'].median(), 2),
}])
summary_df


,profile,attempts,shot_rate_%,shot_box_rate_%,goals_total,goal_rate_%,box_entry_rate_%,final_third_entry_rate_%,median_duration_s,median_in_play_duration_s,median_progression_x,median_passes,median_progression_goal_m,median_start_goal_distance_m,median_required_speed_mps,median_avg_progress_speed_mps,median_speed_ratio
0,balanced_v2_scaled_tracking_coords,746,22.39,17.29,19,2.55,56.84,81.37,7.75,7.37,39.43,2.0,38.38,63.0,3.47,4.57,1.36


## Start-Zone Distribution
Check where the extracted counterattacks start on the pitch.


In [8]:
zone_df = (
    attempts_df.groupby('start_zone')
    .size()
    .rename('n')
    .reset_index()
)
if not zone_df.empty:
    zone_df['pct'] = 100 * zone_df['n'] / zone_df['n'].sum()
zone_df.sort_values('n', ascending=False)


,start_zone,n,pct
2,middle_third,327,43.833780
1,defensive_third,280,37.533512
0,attacking_third,139,18.632708


## Save Derived Outputs
Write the attempts and summary tables used by the later modelling and analysis steps.


In [9]:
output_dir = Path('Data/derived')
output_dir.mkdir(parents=True, exist_ok=True)

attempts_df.to_csv(output_dir / 'counterattack_attempts_balanced_v2_scaled_tracking_coords.csv', index=False)
summary_df.to_csv(output_dir / 'counterattack_profile_summary_tracking_coords.csv', index=False)
zone_df.to_csv(output_dir / 'counterattack_start_zone_tracking_coords.csv', index=False)

print('Saved:')
print('-', output_dir / 'counterattack_attempts_balanced_v2_scaled_tracking_coords.csv')
print('-', output_dir / 'counterattack_profile_summary_tracking_coords.csv')
print('-', output_dir / 'counterattack_start_zone_tracking_coords.csv')


Saved:
- Data/derived/counterattack_attempts_balanced_v2_scaled_tracking_coords.csv
- Data/derived/counterattack_profile_summary_tracking_coords.csv
- Data/derived/counterattack_start_zone_tracking_coords.csv
